In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [2]:
import pandas as pd
import numpy as np

In [3]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/test.csv")

In [4]:
train.shape

(699635, 23)

In [5]:
test.shape

(299844, 22)

In [6]:
train.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 699635 non-null  int64  
 1   Age                                699635 non-null  int64  
 2   Flight Distance                    699635 non-null  int64  
 3   Inflight wifi service              699635 non-null  int64  
 4   Departure/Arrival time convenient  699635 non-null  int64  
 5   Ease of Online booking             699635 non-null  int64  
 6   Gate location                      699635 non-null  int64  
 7   Food and drink                     699635 non-null  int64  
 8   Online boarding                    699635 non-null  int64  
 9   Seat comfort                       699635 non-null  int64  
 10  Inflight entertainment             699635 non-null  int64  
 11  On-board service                   6996

In [8]:
train.isnull().sum()

id                                     0
Age                                    0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes             292
Gender                                 0
Customer Type                          0
Type of Travel                         0
Class                                  0
satisfaction                           0
dtype: int64

In [9]:
train["satisfaction"].value_counts(normalize=True)

satisfaction
False    0.556427
True     0.443573
Name: proportion, dtype: float64

In [10]:
y = train["satisfaction"].astype(int)
X = train.drop(columns=["id", "satisfaction"])
X_test = test.drop(columns=["id"])

In [11]:
cat_cols = X.select_dtypes("object").columns.tolist()

In [12]:
print("Categorical:", cat_cols)

Categorical: ['Gender', 'Customer Type', 'Type of Travel', 'Class']


In [13]:
for c in cat_cols:
    X[c] = X[c].astype("category")
    X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

print(X.shape, X_test.shape)

(699635, 21) (299844, 21)


In [14]:
import re

def clean(name):
    return re.sub(r"[^0-9a-zA-Z_]", "_", name)

X.columns = [clean(c) for c in X.columns]
X_test.columns = [clean(c) for c in X_test.columns]

print(X.columns.tolist())

['Age', 'Flight_Distance', 'Inflight_wifi_service', 'Departure_Arrival_time_convenient', 'Ease_of_Online_booking', 'Gate_location', 'Food_and_drink', 'Online_boarding', 'Seat_comfort', 'Inflight_entertainment', 'On_board_service', 'Leg_room_service', 'Baggage_handling', 'Checkin_service', 'Cleanliness', 'Departure_Delay_in_Minutes', 'Arrival_Delay_in_Minutes', 'Gender', 'Customer_Type', 'Type_of_Travel', 'Class']


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

In [16]:
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [17]:
model = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,        
)

In [18]:
model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="auc",
    callbacks=[early_stopping(100), log_evaluation(200)],
)

Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.957543	valid_0's binary_logloss: 0.230553
[400]	valid_0's auc: 0.957802	valid_0's binary_logloss: 0.229883
Early stopping, best iteration is:
[413]	valid_0's auc: 0.957814	valid_0's binary_logloss: 0.229865


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=2000,
               num_leaves=64, random_state=42, subsample=0.8, subsample_freq=1,
               verbose=-1)

In [19]:
pred = model.predict_proba(X_va)[:, 1]
print("Validation AUC:", roc_auc_score(y_va, pred))

Validation AUC: 0.9578136358141169


In [20]:
test_pred = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": test["id"],
    "satisfaction": test_pred
})
submission.to_csv("submission.csv", index=False)

print(submission.shape)
submission.head()

(299844, 2)


,id,satisfaction
0,699635,0.226588
1,699636,0.856372
2,699637,0.921086
3,699638,0.063522
4,699639,0.043715


In [21]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_lgb = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))

In [22]:
for fold, (tr, va) in enumerate(skf.split(X, y)):
    m = LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.05,
        num_leaves=64,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
    )
    m.fit(
        X.iloc[tr], y.iloc[tr],
        eval_set=[(X.iloc[va], y.iloc[va])],
        eval_metric="auc",
        callbacks=[early_stopping(100), log_evaluation(0)],
    )
    oof_lgb[va] = m.predict_proba(X.iloc[va])[:, 1]
    test_lgb += m.predict_proba(X_test)[:, 1] / 5
    print(f"Fold {fold+1} AUC:", roc_auc_score(y.iloc[va], oof_lgb[va]))

print("CV AUC:", roc_auc_score(y, oof_lgb))

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[629]	valid_0's auc: 0.958924	valid_0's binary_logloss: 0.227181
Fold 1 AUC: 0.9589239107514513
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[409]	valid_0's auc: 0.957702	valid_0's binary_logloss: 0.229753
Fold 2 AUC: 0.9577018644448797
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[555]	valid_0's auc: 0.959632	valid_0's binary_logloss: 0.224675
Fold 3 AUC: 0.9596318438232387
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[511]	valid_0's auc: 0.958901	valid_0's binary_logloss: 0.226648
Fold 4 AUC: 0.9589011793215498
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[406]	valid_0's auc: 0.959028	valid_0's binary_logloss: 0.226952
Fold 5 AUC: 0.9590283806320773
CV AUC: 0.9588350529888091


In [23]:
submission_cv = pd.DataFrame({
    "id": test["id"],
    "satisfaction": test_lgb
})
submission_cv.to_csv("submission.csv", index=False)

print(submission_cv.shape)
submission_cv.head()

(299844, 2)


,id,satisfaction
0,699635,0.234284
1,699636,0.882085
2,699637,0.923699
3,699638,0.066440
4,699639,0.050952


In [24]:
from xgboost import XGBClassifier

oof_xgb = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

X_x = X.copy()
X_test_x = X_test.copy()

for fold, (tr, va) in enumerate(skf.split(X_x, y)):
    m = XGBClassifier(
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        enable_categorical=True,
        eval_metric="auc",
        early_stopping_rounds=100,
        random_state=42,
        n_jobs=-1,
    )
    m.fit(
        X_x.iloc[tr], y.iloc[tr],
        eval_set=[(X_x.iloc[va], y.iloc[va])],
        verbose=False,
    )
    oof_xgb[va] = m.predict_proba(X_x.iloc[va])[:, 1]
    test_xgb += m.predict_proba(X_test_x)[:, 1] / 5
    print(f"Fold {fold+1} AUC:", roc_auc_score(y.iloc[va], oof_xgb[va]))

print("XGB CV AUC:", roc_auc_score(y, oof_xgb))

Fold 1 AUC: 0.9587811772236503
Fold 2 AUC: 0.9576362749855263
Fold 3 AUC: 0.95953106840303
Fold 4 AUC: 0.9587509584986578
Fold 5 AUC: 0.9589477701250847
XGB CV AUC: 0.9587201355547927


In [25]:
from scipy.stats import rankdata

blend = (rankdata(oof_lgb) + rankdata(oof_xgb)) / 2
print("Blend CV AUC:", roc_auc_score(y, blend))

Blend CV AUC: 0.9590118593316179


In [26]:
test_blend = (rankdata(test_lgb) + rankdata(test_xgb)) / 2
test_blend = test_blend / test_blend.max()  

submission_blend = pd.DataFrame({
    "id": test["id"],
    "satisfaction": test_blend
})
submission_blend.to_csv("submission.csv", index=False)

print(submission_blend.shape)
submission_blend.head()

(299844, 2)


,id,satisfaction
0,699635,0.516962
1,699636,0.619496
2,699637,0.637169
3,699638,0.423368
4,699639,0.394633
